# Convert TSV to JSON with chDB

In-process ClickHouse SQL, no server. Run `./generate.sh` first to create `./data/events.tsv`.

Requires: `pip install chdb`

In [ ]:
import os
import chdb

os.chdir("data")

## 1. TSV -> JSONEachRow (one object per line / NDJSON)

In [ ]:
ndjson = chdb.query("SELECT * FROM file('events.tsv') LIMIT 3", "JSONEachRow")
print(str(ndjson).strip())

# Write the full file out.
with open("events_chdb.jsonl", "w") as f:
    f.write(str(chdb.query("SELECT * FROM file('events.tsv')", "JSONEachRow")))

## 2. TSV -> a single JSON array

`output_format_json_array_of_rows = 1` turns the line-delimited stream into one JSON array.

In [ ]:
arr = chdb.query(
    "SELECT * FROM file('events.tsv') LIMIT 3 "
    "SETTINGS output_format_json_array_of_rows = 1",
    "JSONEachRow",
)
print(str(arr).strip())

with open("events_chdb.json", "w") as f:
    f.write(str(chdb.query(
        "SELECT * FROM file('events.tsv') "
        "SETTINGS output_format_json_array_of_rows = 1",
        "JSONEachRow",
    )))

## 3. Transform on the way out (filter + rename)

In [ ]:
print(str(chdb.query(
    "SELECT event_date, country, upper(action) AS action_upper, value "
    "FROM file('events.tsv') WHERE action = 'purchase' "
    "ORDER BY value DESC LIMIT 3",
    "JSONEachRow",
)).strip())